In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score

In [ ]:
!pip install xgboost

In [2]:
def split_scalar(indep_x1,dep_y1):    
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split
    sc=StandardScaler()
    x_train,x_test,y_train,y_test=train_test_split(indep_x1,dep_y1,test_size=0.30,random_state=0)
    x_train_pp=sc.fit_transform(x_train)
    x_test_pp=sc.transform(x_test)
    return x_train_pp,x_test_pp,y_train, y_test,sc

def mlr(x_train_pp,x_test_pp,y_train,y_test):    
    from sklearn.linear_model import LinearRegression
    reg=LinearRegression()
    reg.fit(x_train_pp,y_train)
    y_pred = reg.predict(x_test_pp)
    re=r2_score(y_test,y_pred)
    return re,reg

def svm(x_train_pp,x_test_pp,y_train,y_test):    
    from sklearn.svm import SVR
    param_grid={'kernel':['rbf','poly','sigmoid','linear'],'C':[10,1000,1000,2000,3000]}
    grid=GridSearchCV(SVR(),param_grid,refit=True,verbose=3,n_jobs=-1)
    grid.fit(x_train_pp,y_train)
    y_pred = grid.predict(x_test_pp)
    re= r2_score(y_test, y_pred)
    return re,grid.best_estimator_

def dt(x_train_pp,x_test_pp,y_train,y_test):    
    from sklearn.tree import DecisionTreeRegressor
    param_grid = {'criterion':    ['squared_error', 'absolute_error', 'friedman_mse'],  # ✅ fixed
        'max_features': ['sqrt', 'log2', None],                               # ✅ fixed
        'splitter':     ['best', 'random']}
    grid = GridSearchCV(DecisionTreeRegressor(), param_grid, refit=True, cv=5, n_jobs=-1)
    grid.fit(x_train_pp,y_train)
    y_pred = grid.predict(x_test_pp)
    re= r2_score(y_test, y_pred)
    return re,grid.best_estimator_
    
def rf(x_train_pp,x_test_pp,y_train,y_test):    
    from sklearn.ensemble import RandomForestRegressor
    param_grid = {'criterion':    ['squared_error', 'absolute_error'],                  # ✅ fixed
        'max_features': ['sqrt', 'log2', None],                               # ✅ fixed
        'n_estimators': [10, 100]}
    grid = GridSearchCV(RandomForestRegressor(), param_grid, refit=True, cv=5, n_jobs=-1)
    grid.fit(x_train_pp,y_train)
    y_pred = grid.predict(x_test_pp)
    re= r2_score(y_test, y_pred)
    return re,grid.best_estimator_

def xg(x_train_pp,x_test_pp,y_train,y_test):    

    from xgboost import XGBRegressor
    param_grid = {'n_estimators':  [100, 200],'max_depth':[3, 5], 'learning_rate': [0.01, 0.1], 'subsample': [0.8, 1.0] }
    grid= GridSearchCV(XGBRegressor(random_state=42),param_grid,scoring='r2',cv=5,n_jobs=-1,refit=True)
    grid.fit(x_train_pp,y_train)
    y_pred = grid.predict(x_test_pp)
    re= r2_score(y_test, y_pred)
    return re,grid.best_estimator_


def Reg_rscore_result(regmlr,regsvm,regdt,regrf,regxg): 
    
    dataframe=pd.DataFrame(index=['rscore'],columns=['MLR','SVM','DecisionTree','RandomForest','XGBoosting'])

    for number,idex in enumerate(dataframe.index):
        
        dataframe.loc[idex, 'MLR']          = regmlr[number]
        dataframe.loc[idex, 'SVM']          = regsvm[number]
        dataframe.loc[idex, 'DecisionTree'] = regdt[number]
        dataframe.loc[idex, 'RandomForest'] = regrf[number]
        dataframe.loc[idex, 'XGBoosting']   = regxg[number]
    return dataframe








In [3]:
df=pd.read_csv('ScreenTime_vs_MentalWellness.csv')
df = df.drop(['Unnamed: 15','user_id'],axis=1)
df


,age,gender,occupation,work_mode,screen_time_hours,work_screen_hours,leisure_screen_hours,sleep_hours,sleep_quality_1_5,stress_level_0_10,productivity_0_100,exercise_minutes_per_week,social_hours_per_week,mental_wellness_index_0_100
0,33,Female,Employed,Remote,10.79,5.44,5.35,6.63,1,9.3,44.7,127,0.7,9.3
1,28,Female,Employed,In-person,7.40,0.37,7.03,8.05,3,5.7,78.0,74,2.1,56.2
2,35,Female,Employed,Hybrid,9.78,1.09,8.69,6.48,1,9.1,51.8,67,8.0,3.6
3,42,Male,Employed,Hybrid,11.13,0.56,10.57,6.89,1,10.0,37.0,0,5.7,0.0
4,28,Male,Student,Remote,13.22,4.09,9.13,5.79,1,10.0,38.5,143,10.1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,26,Female,Student,Remote,6.43,2.99,3.44,7.75,1,5.9,64.6,252,7.6,39.3
396,16,Male,Self-employed,Remote,9.59,5.44,4.15,5.57,1,10.0,47.4,99,7.0,3.5
397,40,Male,Student,Remote,8.72,2.36,6.36,7.56,1,9.4,57.3,193,10.1,6.6
398,29,Female,Retired,Hybrid,5.04,0.94,4.10,7.32,1,7.1,63.6,97,12.1,21.0


In [4]:
df=pd.get_dummies(df,drop_first=True).astype(int)
df.isnull().sum()
df = df.apply(pd.to_numeric, errors='coerce')
df

,age,screen_time_hours,work_screen_hours,leisure_screen_hours,sleep_hours,sleep_quality_1_5,stress_level_0_10,productivity_0_100,exercise_minutes_per_week,social_hours_per_week,mental_wellness_index_0_100,gender_Male,gender_Non-binary/Other,occupation_Retired,occupation_Self-employed,occupation_Student,occupation_Unemployed,work_mode_In-person,work_mode_Remote
0,33,10,5,5,6,1,9,44,127,0,9,0,0,0,0,0,0,0,1
1,28,7,0,7,8,3,5,78,74,2,56,0,0,0,0,0,0,1,0
2,35,9,1,8,6,1,9,51,67,8,3,0,0,0,0,0,0,0,0
3,42,11,0,10,6,1,10,37,0,5,0,1,0,0,0,0,0,0,0
4,28,13,4,9,5,1,10,38,143,10,0,1,0,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,26,6,2,3,7,1,5,64,252,7,39,0,0,0,0,1,0,0,1
396,16,9,5,4,5,1,10,47,99,7,3,1,0,0,1,0,0,0,1
397,40,8,2,6,7,1,9,57,193,10,6,1,0,0,0,1,0,0,1
398,29,5,0,4,7,1,7,63,97,12,21,0,0,1,0,0,0,0,0


In [5]:
indep_x=df[['screen_time_hours', 'sleep_quality_1_5', 'stress_level_0_10','productivity_0_100','exercise_minutes_per_week', 'social_hours_per_week']]
dep_y=df['mental_wellness_index_0_100']

In [6]:
indep_x1=indep_x
dep_y1=dep_y


In [7]:
relist=[1]
regmlr=[]
regsvm=[]
regdt=[]
regrf=[]
regxg=[]





In [8]:

for i in relist:   
    x_train_pp, x_test_pp, y_train, y_test,sc=split_scalar(indep_x,dep_y) 
    remlr,model_mlr=mlr(x_train_pp,x_test_pp,y_train,y_test)
    regmlr.append(remlr)
    
    resvm,model_svm=svm(x_train_pp,x_test_pp,y_train,y_test)
    regsvm.append(resvm)
    
    redt,model_dt=dt(x_train_pp,x_test_pp,y_train,y_test)
    regdt.append(redt)
    
    rerf,model_rf=rf(x_train_pp,x_test_pp,y_train,y_test)
    regrf.append(rerf)
    
    rexg,model_xg=xg(x_train_pp,x_test_pp,y_train,y_test)
    regxg.append(rexg)

result=Reg_rscore_result(regmlr,regsvm,regdt,regrf,regxg)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


In [9]:
result

,MLR,SVM,DecisionTree,RandomForest,XGBoosting
rscore,0.935449,0.935334,0.86049,0.913009,0.911224


In [10]:
resultMLR=remlr
resultMLR

0.9354493728005422

In [11]:
import pickle
# Step 2 — Create a dictionary of all models
all_models = {
    'MLR':          regmlr,
    'SVM':          regsvm,
    'DecisionTree': regdt,
    'RandomForest': regrf,
    'XGBoosting':   regxg}
# Step 3 — Automatically pick best model by highest r2 score
best_algo = result.loc['rscore'].astype(float).idxmax()
best_model = all_models[best_algo]
print(f"Best Algorithm: {best_algo}")

Best Algorithm: MLR


In [12]:
filename="finalized_screentime vs Mentalwellness_prepro X only.sav"

pickle.dump(sc,open(filename,'wb'))

In [13]:
input=pickle.load(open("finalized_screentime vs Mentalwellness_prepro X only.sav",'rb'))

In [14]:

filename1="finalized_screentime vs Mentalwellness_MLR.sav"

pickle.dump(model_mlr,open(filename1,'wb'))

In [15]:
loaded_model=pickle.load(open("finalized_screentime vs Mentalwellness_MLR.sav",'rb'))

In [16]:
print(type(loaded_model))

<class 'sklearn.linear_model._base.LinearRegression'>


In [17]:
preinput = pd.DataFrame([[9, 2, 5, 50, 200, 1]],
                         columns=['screen_time_hours', 'sleep_quality_1_5',
                                  'stress_level_0_10', 'productivity_0_100',
                                  'exercise_minutes_per_week', 'social_hours_per_week'])
preinput_scaled=sc.transform(preinput)
Mental_wellness_result=loaded_model.predict(preinput_scaled)
Mental_wellness_result

array([40.49860463])